# Dataset Rebalancing Impact Analysis

This notebook compares the original `curated_data.jsonl` against the newly mitigated `curated_data_rebalanced.jsonl` to visualize the effectiveness of the targeted down-sampling and scrambling. It analyzes:
- Frequency reduction of the targeted overrepresented prefixes (`Aah`, `Aiyoh`, `Eh`, `Seriously lah`, `Actually`).
- Impact on the overall word length distribution to assure the persona and length wasn't negatively skewed.


In [ ]:
import json
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Apply cyberpunk/dark theme consistent with previous UI if desired, or standard clean.
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams.update({'font.size': 12})


## 1. Load Datasets and Extract Metrics
We parse both JSONL files and extract the word lengths alongside the starting 1-gram and 2-grams of the assistant's responses.

In [ ]:
def extract_metrics(file_path):
    prefixes = []
    lengths = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                data = json.loads(line)
                
                text = " ".join([t.get("content", "") for t in data.get("conversations", []) if t.get("role") == "assistant"])
                if not text.strip(): continue
                
                # Extract length
                words_raw = text.split()
                lengths.append(len(words_raw))
                
                # Extract 1-gram / 2-gram prefix strictly
                words = [re.sub(r'[^\w\s]', '', w).lower() for w in words_raw[:2] if w]
                if words:
                    prefixes.append(words[0])
                    if len(words) > 1:
                        prefixes.append(f"{words[0]} {words[1]}")
                        
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        
    return prefixes, lengths

print("Loading original data...")
orig_prefixes, orig_lengths = extract_metrics("curated_data.jsonl")

print("Loading rebalanced data...")
rebal_prefixes, rebal_lengths = extract_metrics("curated_data_rebalanced.jsonl")

print(f"Original rows processed: {len(orig_lengths)}")
print(f"Rebalanced rows processed: {len(rebal_lengths)}")


## 2. Prefix Overrepresentation Comparison
Visualizing the exact reduction in frequency of our targeted overrepresented stylings.

In [ ]:
# Target prefixes we mitigated
targets = ['aah', 'aiyoh', 'eh', 'seriously lah', 'actually']

orig_counts = Counter(orig_prefixes)
rebal_counts = Counter(rebal_prefixes)

comp_data = []
for t in targets:
    comp_data.append({"Prefix": t.title(), "Count": orig_counts[t], "Dataset": "Original"})
    comp_data.append({"Prefix": t.title(), "Count": rebal_counts[t], "Dataset": "Rebalanced"})
    
df_comp = pd.DataFrame(comp_data)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_comp, x="Count", y="Prefix", hue="Dataset", palette=["#f20089", "#00f0ff"])
plt.title("Target Prefix Frequencies: Original vs Rebalanced", fontsize=14, fontweight='bold')
plt.xlabel("Frequency (Number of Responses)")
plt.ylabel("Target Prefix")

# Add value text formatting
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.text(width + 100, p.get_y() + p.get_height() / 2, f'{int(width)}', va='center')
        
plt.show()


## 3. Impact on Word Count Distribution
Checking if our mitigation strategies (like deleting or moving prefixes inside sentences) drastically altered the length characteristics of the dataset. Because we swapped or shifted words linearly, we should see highly overlapping distributions.

In [ ]:
df_len = pd.DataFrame({
    "Word Count": orig_lengths + rebal_lengths,
    "Dataset": ["Original"] * len(orig_lengths) + ["Rebalanced"] * len(rebal_lengths)
})

# Filter out extreme outliers for visualization purposes
df_len_filtered = df_len[df_len["Word Count"] < df_len["Word Count"].quantile(0.95)]

plt.figure(figsize=(12, 6))
sns.histplot(data=df_len_filtered, x="Word Count", hue="Dataset", kde=True, bins=40, palette=["#f20089", "#00f0ff"], alpha=0.5)
plt.title("Word Count Distribution Comparison (Filtered to 95th Percentile)", fontsize=14, fontweight='bold')
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.show()

print("--- Descriptive Statistics of Dataset Responses ---")
print(df_len.groupby("Dataset")["Word Count"].describe()[["mean", "std", "min", "50%", "max"]])
